# Baseline Evaluation — gpt-4.1-nano on Recipe Generation

**CIS 5270 Course Project — Team: Akshat, Rashi, Ritika**

Evaluates the **base** `gpt-4.1-nano` model on our recipe generation task

## Pipeline (per spec)

- **Step 0.** Azure client + config
- **Step 1.** Read `rft_val.jsonl` (400 ingredient lists)
- **Step 2.** Create SYSTEM + USER prompts (same as training)
- **Step 3.** LLM call function
- **Step 4.** Evaluation via **LLM-as-judge** (one call per sample). Judge returns:
    - Five bools: `ingredient_compliance`, `num_steps_in_range`, `cook_time_in_range`, `coherence`, `student_friendly`
    - Two descriptive integers: `num_steps` (raw count) and `cook_time_min` (raw minutes)
    - `all_pass = 1` iff the 3 hard-constraint bools are all 1.
    - `llm_score = 0.5 * coherence + 0.5 * student_friendly` ∈ {0, 0.5, 1}.

## Step 0 — Setup

In [ ]:
%pip install -q openai azure-identity python-dotenv tqdm pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 4.4 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional
from dataclasses import dataclass

import pandas as pd
from tqdm import tqdm
from openai import AzureOpenAI

### Azure client

In [ ]:
RESOURCE_GROUP  = "cis-5270-team-14"
OPENAI_API_KEY  = ""
OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ''

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)
print("Connected to Azure OpenAI")

Connected to Azure OpenAI


### Config — matches the spec

- model: `gpt-4.1-nano`
- temperature: **1.0**
- max_tokens: **1024** (generation) / **512** (judge)
- eval set: `baseline_val.jsonl` (~400 samples)

In [ ]:
@dataclass
class EvalConfig:
    # Constraints
    min_steps: int = 3
    max_steps: int = 7
    max_cook_minutes: int = 15

    # Paths.
    val_path: str = "sft_val.jsonl"
    out_dir:  str = "results"

    # Inference hyperparameters
    temperature: float = 1.0
    max_tokens_gen: int   = 1024
    max_tokens_judge: int = 512

    model: str = "1-nano-2025-04-14-supervised-fine-tuning-"
    # Short tag used in output filenames (base, sft, dpo, rft).
    tag: str = "sft-beta"

    # Judge model
    judge_model: str = "gpt-4.1-mini"

    # Parallelism + retries.
    workers: int = 1


    inter_sample_sleep: float = 7
    max_retries: int = 3
    retry_base_sleep: float = 6.0


ECFG = EvalConfig()
Path(ECFG.out_dir).mkdir(exist_ok=True)
print(f"Evaluating model = {ECFG.model!r} (tag = {ECFG.tag!r})")
print(f"Constraints: {ECFG.min_steps}-{ECFG.max_steps} steps, <= {ECFG.max_cook_minutes} min")
print(f"Output dir:  {ECFG.out_dir}/")

Evaluating model = '1-nano-2025-04-14-supervised-fine-tuning-' (tag = 'sft-beta')
Constraints: 3-7 steps, <= 15 min
Output dir:  results/


## Step 1 — Read `baseline_val.jsonl`

RFT records already carry `ingredients`, `messages`, and `answer` as structured fields, so we don't need any regex parsing.

In [ ]:
import re

def load_eval_set(path: str) -> list[dict]:
    samples = []
    with open(path) as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            system_msg = next(m["content"] for m in rec["messages"] if m["role"] == "system")
            user_msg   = next(m["content"] for m in rec["messages"] if m["role"] == "user")

            if "ingredients" in rec:
                ingredients = rec["ingredients"]
            else:
                # User prompt pattern:
                m = re.search(
                    r"using these ingredients:\s*(.+?)(?:\.\s*Keep it|\.\s*$|$)",
                    user_msg,
                    re.IGNORECASE | re.DOTALL,
                )
                if m:
                    raw = m.group(1).strip().rstrip(".")
                    ingredients = [ing.strip() for ing in raw.split(",") if ing.strip()]
                else:
                    ingredients = []

            samples.append({
                "id":               i,
                "ingredients":      ingredients,
                "system_prompt":    system_msg,
                "user_prompt":      user_msg,
                "reference_recipe": rec.get("answer", ""),
            })
    return samples

eval_set = load_eval_set(ECFG.val_path)
print(f"Loaded {len(eval_set)} eval samples from {ECFG.val_path}")
print("\nFirst sample:")
print(f"  ingredients: {eval_set[0]['ingredients']}")
print(f"  user prompt: {eval_set[0]['user_prompt'][:120]}...")

Loaded 400 eval samples from sft_val.jsonl

First sample:
  ingredients: ['red lentils', 'turmeric', 'garlic', 'basmati rice']
  user prompt: Create a quick, student-friendly recipe using these ingredients: red lentils, turmeric, garlic, basmati rice. Keep it un...


## Step 2 — Prompts

The prompts are stored on the record itself (system + user, written during data generation). Using them verbatim is what keeps the comparison apples-to-apples across base / SFT / DPO / RFT — every model sees the exact same input.

In [ ]:
print("SYSTEM:")
print("  " + eval_set[0]["system_prompt"])
print("\nUSER (example):")
print("  " + eval_set[0]["user_prompt"])

SYSTEM:
  You are a recipe assistant for college students. Generate quick recipes (under 15 minutes) with 3-7 clear numbered steps, using only the ingredients provided.

USER (example):
  Create a quick, student-friendly recipe using these ingredients: red lentils, turmeric, garlic, basmati rice. Keep it under 15 minutes with 3-7 clear steps.


## Step 3 — LLM call function

One helper for both generation and judging. Retries on transient errors; returns `None` on permanent failure so the row still gets logged (never silently drops samples).

In [ ]:
import random

def chat_call(
    model: str,
    system: str,
    user: str,
    temperature: float = ECFG.temperature,
    max_tokens: int = ECFG.max_tokens_gen,
    max_retries: int = ECFG.max_retries,
) -> Optional[str]:
    for attempt in range(max_retries):
        try:
            _log_request(model)
            resp = openai_client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user",   "content": user},
                ],
                temperature=temperature,
                max_completion_tokens=max_tokens,
            )
            return resp.choices[0].message.content
        except Exception as e:
            err = str(e)
            is_rate_limit = "429" in err or "too_many_requests" in err.lower()
            if attempt == max_retries - 1:
                print(f"  chat_call failed after {max_retries} attempts: {e}")
                return None
            # Exponential backoff
            sleep = (ECFG.retry_base_sleep * (2 ** attempt) + random.uniform(0, 1))
            if is_rate_limit:
                sleep = max(sleep, 30.0)
            time.sleep(sleep)
    return None

In [ ]:
import threading
from collections import deque

_log_lock   = threading.Lock()
_req_log    = deque()

def _log_request(model_label: str):
    now = time.time()
    with _log_lock:
        _req_log.append((now, model_label))

def print_rpm_summary():
    with _log_lock:
        entries = list(_req_log)

    if not entries:
        print("No requests logged yet.")
        return

    # Group into 60-second buckets
    t0 = entries[0][0]
    buckets: dict[int, dict[str, int]] = {}
    for ts, model in entries:
        bucket = int((ts - t0) // 60)
        buckets.setdefault(bucket, {})
        buckets[bucket][model] = buckets[bucket].get(model, 0) + 1

    gen_model   = ECFG.model
    judge_model = ECFG.judge_model

    print(f"\n{'Min':>4}  {'Generator ('+gen_model+')':>40}  {'Judge ('+judge_model+')':>20}  {'Total':>6}")
    print("-" * 80)
    total_gen = total_judge = 0
    for bucket in sorted(buckets):
        g = buckets[bucket].get(gen_model, 0)
        j = buckets[bucket].get(judge_model, 0)
        total_gen   += g
        total_judge += j
        over = "OVER 10" if (g + j) > 10 else ""
        print(f"{bucket:>4}  {g:>40}  {j:>20}  {g+j:>6}{over}")
    print("-" * 80)
    print(f"{'TOT':>4}  {total_gen:>40}  {total_judge:>20}  {total_gen+total_judge:>6}")

## Step 4 — Evaluation via LLM-as-judge

One judge call per sample. The judge reads the ingredient list + the generated recipe and returns:

**Hard-constraint bools** (feed into `all_pass`):
- **`ingredient_compliance`** — every provided ingredient is used, no unauthorized ingredients introduced (pantry staples allowed).
- **`num_steps_in_range`** — the recipe has between `min_steps` and `max_steps` numbered steps.
- **`cook_time_in_range`** — the recipe's total cook time is `≤ max_cook_minutes`.

**Soft-quality bools** (feed into `llm_score`):
- **`coherence`** — the recipe describes a plausible, edible dish (not a word-salad).
- **`student_friendly`** — realistic for a dorm kitchen, no specialty equipment.

**Descriptive integers** (raw values for the 'average recipe length' metric in the proposal):
- **`num_steps`** — raw count of steps in the recipe.
- **`cook_time_min`** — raw total cook time in minutes.

**Rollups** (computed by us from the judge output):
- `all_pass = 1` iff `ingredient_compliance`, `num_steps_in_range`, and `cook_time_in_range` are all 1.
- `llm_score = 0.5 * coherence + 0.5 * student_friendly` ∈ {0, 0.5, 1}.

The judge runs at `temperature=0` to keep outputs deterministic — same recipe, same judgment.

In [ ]:
JUDGE_SYSTEM_TEMPLATE = """You are a strict evaluator of college-student recipes.

You will see (1) an ingredient list, (2) a generated recipe, and (3) the constraint limits. Evaluate ALL of the following and return JSON.

**Hard constraints (return each as true/false):**

1. `ingredient_compliance`: true iff every provided ingredient is used in the recipe AND the recipe does NOT introduce ingredients that weren't provided. Common pantry staples (water, salt, pepper, cooking oil, butter) are always allowed. Minor descriptor differences are fine (e.g. the list has 'egg' and the recipe says 'eggs', or 'onion' vs 'chopped onion').
2. `num_steps_in_range`: true iff the recipe has between {min_steps} and {max_steps} numbered steps (inclusive). Count steps by reading the recipe, not by counting lines.
3. `cook_time_in_range`: true iff the recipe's total cook time is {max_cook_minutes} minutes or less. If no explicit time is given, infer it from the steps and mark false if you can't reasonably estimate.

**Soft-quality constraints (return each as true/false):**

4. `coherence`: true iff the recipe describes a plausible, edible dish from these ingredients — not a random sequence of actions or a word-salad.
5. `student_friendly`: true iff a college student could reasonably make this in a dorm kitchen. No specialty equipment, no advanced techniques (e.g. sous vide, tempering chocolate); instructions clear enough for a beginner.

**Descriptive integers:**

6. `num_steps`: exact count of numbered steps in the recipe (integer). If unclear, give your best estimate.
7. `cook_time_min`: total cook time in minutes (integer). If no explicit total is given, estimate from the steps. Use 0 if impossible to estimate.

Return ONLY a JSON object with exactly these keys and no extra text:
{{
  "ingredient_compliance": <true|false>,
  "num_steps_in_range": <true|false>,
  "cook_time_in_range": <true|false>,
  "coherence": <true|false>,
  "student_friendly": <true|false>,
  "num_steps": <integer>,
  "cook_time_min": <integer>,
  "reason": "<one short sentence>"
}}"""

JUDGE_SYSTEM = JUDGE_SYSTEM_TEMPLATE.format(
    min_steps=ECFG.min_steps,
    max_steps=ECFG.max_steps,
    max_cook_minutes=ECFG.max_cook_minutes,
)

def judge_recipe(ingredients: list[str], recipe: str) -> Optional[dict]:
    """Run the judge on one (ingredient list, recipe) pair. Returns a dict of 5 bools
    + 2 raw ints + judge_reason + rollups (all_pass, llm_score). Returns None if the
    judge call fails or returns unparseable JSON."""
    user = (
        f"Ingredient list: {', '.join(ingredients)}\n\n"
        f"Recipe:\n{recipe}\n\n"
        f"Constraint limits: {ECFG.min_steps}-{ECFG.max_steps} steps, <= {ECFG.max_cook_minutes} minutes.\n\n"
        f"Evaluate and return JSON."
    )
    raw = chat_call(
        model=ECFG.judge_model,
        system=JUDGE_SYSTEM,
        user=user,
        temperature=0.0,                      # deterministic judging
        max_tokens=ECFG.max_tokens_judge,
    )
    if raw is None:
        return None
    try:
        cleaned = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
        obj = json.loads(cleaned)

        # Extract the five bools.
        ic  = int(bool(obj.get("ingredient_compliance", False)))
        ns_ok = int(bool(obj.get("num_steps_in_range", False)))
        ct_ok = int(bool(obj.get("cook_time_in_range", False)))
        coh = int(bool(obj.get("coherence", False)))
        sf  = int(bool(obj.get("student_friendly", False)))

        def _as_int(v):
            try: return int(v)
            except Exception: return None
        num_steps     = _as_int(obj.get("num_steps"))
        cook_time_min = _as_int(obj.get("cook_time_min"))

        return {
            # Hard constraints
            "ingredient_compliance": ic,
            "num_steps_in_range":    ns_ok,
            "cook_time_in_range":    ct_ok,
            # Soft quality
            "coherence":             coh,
            "student_friendly":      sf,
            # Descriptive
            "num_steps":             num_steps,
            "cook_time_min":         cook_time_min,
            # Rollups
            "all_pass":              int(ic == 1 and ns_ok == 1 and ct_ok == 1),
            "llm_score":             0.5 * coh + 0.5 * sf,   # in {0, 0.5, 1}
            # Free-text explanation for debugging
            "judge_reason":          obj.get("reason", ""),
        }
    except Exception:
        return None

## Step 5 — Run on all samples

For each sample: `Prompt → LLM call → Evaluation → append to JSONL` with `prompt`, `response`, and both evaluation blocks.

In [ ]:
def evaluate_one(sample: dict) -> dict:
    """Generate + judge one sample. Never raises."""
    print("Evaluating smaple")
    recipe = chat_call(ECFG.model, sample["system_prompt"], sample["user_prompt"])

    if recipe is None:
        return {
            "id":                sample["id"],
            "ingredients":       sample["ingredients"],
            "prompt":            {"system": sample["system_prompt"], "user": sample["user_prompt"]},
            "response":          None,
            "generation_failed": True,
            "evaluation":        None,
        }

    judgment = judge_recipe(sample["ingredients"], recipe)
    print("Evaluating judge")

    return {
        "id":                sample["id"],
        "ingredients":       sample["ingredients"],
        "prompt":            {"system": sample["system_prompt"], "user": sample["user_prompt"]},
        "response":          recipe,
        "generation_failed": False,
        # judgment is None iff the judge call failed; otherwise it's a dict with all
        # bools, raw ints, all_pass, llm_score, and judge_reason.
        "evaluation":        judgment,
    }

In [ ]:
def run_evaluation(samples: list[dict]) -> list[dict]:
    print(f"\n=== Evaluating model={ECFG.model!r} on {len(samples)} samples (tag={ECFG.tag!r}) ===")
    rows = []
    with ThreadPoolExecutor(max_workers=ECFG.workers) as pool:
        futures = []
        for s in samples:
            futures.append(pool.submit(evaluate_one, s))
            time.sleep(ECFG.inter_sample_sleep)

        for fut in tqdm(as_completed(futures), total=len(futures)):
            rows.append(fut.result())
    rows.sort(key=lambda r: r["id"])

    jsonl_path = Path(ECFG.out_dir) / f"{ECFG.tag}_eval.jsonl"
    with open(jsonl_path, "w") as f:
        for r in rows:
            f.write(json.dumps(r, default=str) + "\n")
    print(f"\nWrote per-sample results -> {jsonl_path}")
    return rows

In [ ]:
rows = run_evaluation(eval_set)


=== Evaluating model='1-nano-2025-04-14-supervised-fine-tuning-' on 400 samples (tag='sft-beta') ===
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evaluating judge
Evaluating smaple
Evalu

100%|██████████| 400/400 [00:00<00:00, 211034.16it/s]


Wrote per-sample results -> results/sft-beta_eval.jsonl


## Aggregates for the report

In [ ]:
def aggregate(rows: list[dict]) -> dict:
    n_total = len(rows)
    ok     = [r for r in rows if not r["generation_failed"]]
    judged = [r for r in ok   if r["evaluation"] is not None]

    def _rate(key):
        vals = [r["evaluation"][key] for r in judged]
        return float(sum(vals) / len(vals)) if vals else None

    def _mean(key):
        vals = [r["evaluation"][key] for r in judged if r["evaluation"][key] is not None]
        return float(sum(vals) / len(vals)) if vals else None

    return {
        "model":                  ECFG.model,
        "tag":                    ECFG.tag,
        "n_total":                n_total,
        "n_generated_ok":         len(ok),
        "n_judged":               len(judged),
        "generation_fail_rate":   1 - (len(ok) / n_total) if n_total else None,
        "judge_fail_rate":        1 - (len(judged) / len(ok)) if ok else None,
        # Hard constraints (bool rates) — all from the judge
        "ingredient_compliance_rate": _rate("ingredient_compliance"),
        "num_steps_in_range_rate":    _rate("num_steps_in_range"),
        "cook_time_in_range_rate":    _rate("cook_time_in_range"),
        # Proposal metric: binary constraint satisfaction rate (all 3 hard constraints met)
        "all_pass_rate":              _rate("all_pass"),
        # Proposal metric: average recipe length (raw counts from the judge)
        "avg_num_steps":              _mean("num_steps"),
        "avg_cook_time_min":          _mean("cook_time_min"),
        # Soft-quality metrics (per spec: llm_score uses only these two)
        "coherence_rate":             _rate("coherence"),
        "student_friendly_rate":      _rate("student_friendly"),
        # Proposal metric: preference satisfaction score
        "llm_score_avg":              _mean("llm_score"),
    }

metrics = aggregate(rows)

summary_path = Path(ECFG.out_dir) / "{ECFG.tag}_summary.json"
with open(summary_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"=== Aggregate metrics for tag={ECFG.tag!r} (model={ECFG.model!r}) ===")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:32s} {v:.3f}")
        print(f"  {k:32s} {v}")
print(f"\nSaved -> {summary_path}")

=== Aggregate metrics for tag='sft-beta' (model='1-nano-2025-04-14-supervised-fine-tuning-') ===
  generation_fail_rate             0.000
  generation_fail_rate             0.0
  judge_fail_rate                  0.000
  judge_fail_rate                  0.0
  ingredient_compliance_rate       0.975
  ingredient_compliance_rate       0.975
  num_steps_in_range_rate          0.995
  num_steps_in_range_rate          0.995
  cook_time_in_range_rate          1.000
  cook_time_in_range_rate          1.0
  all_pass_rate                    0.973
  all_pass_rate                    0.9725
  avg_num_steps                    6.055
  avg_num_steps                    6.055
  avg_cook_time_min                12.505
  avg_cook_time_min                12.505
  coherence_rate                   1.000
  coherence_rate                   1.0
  student_friendly_rate            1.000
  student_friendly_rate            1.0
  llm_score_avg                    1.000
  llm_score_avg                    1.0

Saved -> 

## Spot-check — three random outputs

Useful for the "How do your results look?" bullet — gives you concrete examples of what the base model gets right/wrong. Save 1–2 of these for the report as qualitative examples.

In [ ]:
import random
sample_rows = random.sample([r for r in rows if not r["generation_failed"]], min(3, len(rows)))
for r in sample_rows:
    print("=" * 80)
    print(f"Sample #{r['id']}  |  ingredients: {r['ingredients']}")
    print("-" * 80)
    print("RESPONSE:")
    print(r["response"][:600] + ("..." if len(r["response"]) > 600 else ""))
    print()
    ev = r["evaluation"]
    if ev is None:
        print("EVALUATION: judge call failed")
    else:
        print(f"EVALUATION:")
        print(f"  ingredient_compliance = {ev['ingredient_compliance']}  "
              f"num_steps_in_range = {ev['num_steps_in_range']} (raw={ev['num_steps']})  "
              f"cook_time_in_range = {ev['cook_time_in_range']} (raw={ev['cook_time_min']} min)")
        print(f"  coherence = {ev['coherence']}  student_friendly = {ev['student_friendly']}")
        print(f"  all_pass = {ev['all_pass']}  llm_score = {ev['llm_score']}")
        print(f"  reason: {ev['judge_reason']}")
    print()

Sample #51  |  ingredients: ['tortilla española', 'olive oil', 'garlic', 'paprika', 'bread']
--------------------------------------------------------------------------------
RESPONSE:
**Quick Spanish Garlic Mashed Bread** (15 min)

1. Cut the bread into bite-sized cubes.
2. Heat olive oil in a pan over medium heat and add minced garlic; sauté until fragrant.
3. Add the bread cubes and a sprinkle of paprika to the pan and toss to coat with the garlic oil.
4. Continue tossing until the bread is toasted and crispy, about 4-5 minutes.
5. Season with salt and pepper to taste and serve warm.

EVALUATION:
  ingredient_compliance = 0  num_steps_in_range = 1 (raw=5)  cook_time_in_range = 1 (raw=15 min)
  coherence = 1  student_friendly = 1
  all_pass = 0  llm_score = 1.0
  reason: The recipe does not use tortilla española, which is in the ingredient list.

Sample #76  |  ingredients: ['basmati rice', 'chickpeas', 'curry powder', 'onion']
---------------------------------------------------------